# ISC Machine Learning - Analisis Target & Fitur E-Commerce

Notebook ini disiapkan untuk:
1. Eksplorasi data transaksi e-commerce (`dataset.csv`).
2. Analisis korelasi dan **Heatmap** terhadap variabel target (`is_valid` & `is_net`).
3. Heatmap hubungan fitur kategorikal (Metode Pembayaran, Kategori Produk) terhadap rasio target.
4. Pipeline persiapan pemodelan Machine Learning (Feature Engineering, Train/Test Split, Random Forest baseline).

In [ ]:
# 1. Import Library
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Visual configuration
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
print("Semua library berhasil di-load!")

In [ ]:
# 2. Load Dataset
df = pd.read_csv("dataset.csv")
print(f"Dimensi Dataset: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

In [ ]:
# 3. Pengecekan Info Data & Missing Values
print(df.info())
print("\nJumlah Missing Values per Kolom:")
print(df.isnull().sum())

## 4. Analisis Variabel Target
- **`is_valid`**: Status validasi pembayaran customer (1 = Paid / Valid, 0 = Batal / Belum Bayar).
- **`is_net`**: Status transaksi net selesai (1 = Berhasil tanpa retur, 0 = Batal / Retur).
- **`is_gross`**: Flag pembuatan order (semua bernilai 1).

In [ ]:
# Distribusi Target is_valid & is_net
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x="is_valid", ax=axes[0], palette="viridis")
axes[0].set_title("Distribusi Target: is_valid (0: Unpaid/Batal, 1: Valid/Bayar)", fontsize=12)
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha="center", va="center", color="white", fontweight="bold")

sns.countplot(data=df, x="is_net", ax=axes[1], palette="mako")
axes[1].set_title("Distribusi Target: is_net (0: Retur/Gagal, 1: Transaksi Net)", fontsize=12)
for p in axes[1].patches:
    axes[1].annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width() / 2., p.get_height() / 2),
                     ha="center", va="center", color="white", fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Heatmap Korelasi Fitur Terhadap Target
Heatmap berikut menampilkan matriks korelasi fitur numerik dengan fokus utama pada variabel target `is_valid` dan `is_net`.

In [ ]:
# Heatmap Korelasi Semua Fitur Numerik
num_cols = df.select_dtypes(include=["int64", "float64"]).columns
corr_matrix = df[num_cols].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
plt.title("Heatmap Matriks Korelasi Keseluruhan Fitur Numerik", fontsize=14, fontweight="bold")
plt.show()

In [ ]:
# Heatmap Khusus Fokus Target (is_valid & is_net)
target_corr = corr_matrix[["is_valid", "is_net"]].sort_values(by="is_valid", ascending=False)

plt.figure(figsize=(8, 6))
sns.heatmap(target_corr, annot=True, fmt=".3f", cmap="Blues", cbar=True, linewidths=1)
plt.title("Heatmap Korelasi Fitur Terhadap Target (is_valid & is_net)", fontsize=13, fontweight="bold")
plt.show()

## 6. Heatmap Analisis Hubungan Fitur Kategorikal vs Target

In [ ]:
# Heatmap Payment Method vs Target (is_valid %)
pm_crosstab = pd.crosstab(df["payment_method"], df["is_valid"], normalize="index") * 100
pm_crosstab.columns = ["Batal / Unpaid (%)", "Valid / Paid (%)"]

plt.figure(figsize=(10, 6))
sns.heatmap(pm_crosstab.sort_values(by="Valid / Paid (%)", ascending=False), annot=True, fmt=".1f", cmap="YlGnBu", cbar=True)
plt.title("Heatmap Persentase Validasi per Metode Pembayaran (%)", fontsize=13, fontweight="bold")
plt.ylabel("Metode Pembayaran")
plt.show()

In [ ]:
# Heatmap Category vs Target (is_valid %)
cat_crosstab = pd.crosstab(df["category"], df["is_valid"], normalize="index") * 100
cat_crosstab.columns = ["Batal / Unpaid (%)", "Valid / Paid (%)"]

plt.figure(figsize=(10, 6))
sns.heatmap(cat_crosstab.sort_values(by="Valid / Paid (%)", ascending=False), annot=True, fmt=".1f", cmap="Greens", cbar=True)
plt.title("Heatmap Persentase Validasi per Kategori Produk (%)", fontsize=13, fontweight="bold")
plt.ylabel("Kategori Produk")
plt.show()

## 7. Persiapan Fitur & Baseline Pemodelan Machine Learning

In [ ]:
# Feature Engineering Sederhana
df_model = df.copy()
df_model["order_date"] = pd.to_datetime(df_model["order_date"])
df_model["order_month"] = df_model["order_date"].dt.month
df_model["order_dayofweek"] = df_model["order_date"].dt.dayofweek
df_model["has_discount"] = (df_model["discount_amount"] > 0).astype(int)

# Seleksi fitur numerik prediktif
features = [
    "price", "qty_ordered", "before_discount", "discount_amount", "after_discount",
    "cogs", "payment_id", "order_month", "order_dayofweek", "has_discount"
]
target = "is_valid"

X = df_model[features]
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")

In [ ]:
# Baseline Model: Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
print("=== Classification Report ===")
print(classification_report(y_test, y_pred))

# Feature Importance Visual
feat_imp = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(x=feat_imp.values, y=feat_imp.index, palette="Blues_r")
plt.title("Feature Importance - Random Forest", fontsize=13, fontweight="bold")
plt.xlabel("Importance Score")
plt.show()